In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.impute import KNNImputer

In [2]:
# ===== 1. Đọc file =====
input_path = Path(r"D:\User\Tài liệu học\CCDNCTKHDL\6. Smart-PLS\Matrix_wide.csv")
df = pd.read_csv(input_path)

print("Số dòng file gốc:", df.shape[0])
print("Số cột file gốc:", df.shape[1])
print("Các cột:", list(df.columns))

Số dòng file gốc: 60492
Số cột file gốc: 17
Các cột: ['review_id', 'AR', 'AT', 'CS', 'GS', 'HH', 'LV', 'MH', 'NQ', 'OA', 'PE', 'RC', 'RE', 'RP', 'SR', 'VP', 'satisfaction_score']


In [3]:
# ===== 2. Tách cột ID, satisfaction_score và các cột sentiment_score =====
# Code này tự nhận diện Review_ID / review_id để tránh lỗi do khác tên cột.
possible_id_cols = ["Review_ID", "review_id", "ID_review", "id_review"]
id_col = next((col for col in possible_id_cols if col in df.columns), None)

if id_col is None:
    raise ValueError(f"Không tìm thấy cột ID review. Các cột hiện có: {list(df.columns)}")

satisfaction_col = "satisfaction_score"

# Các cột sentiment_score/topic: toàn bộ cột numeric, trừ ID và satisfaction_score.
sentiment_cols = [
    col for col in df.columns
    if col not in [id_col, satisfaction_col]
    and pd.api.types.is_numeric_dtype(df[col])
]

# Các cột sẽ KNN fill: sentiment_score tại từng topic + satisfaction_score nếu tồn tại và là numeric.
cols_to_impute = sentiment_cols.copy()

if satisfaction_col in df.columns:
    if pd.api.types.is_numeric_dtype(df[satisfaction_col]):
        cols_to_impute.append(satisfaction_col)
    else:
        raise ValueError(f"Cột {satisfaction_col} tồn tại nhưng không phải dạng numeric, cần chuyển sang số trước khi KNN fill.")
else:
    print(f"Không tìm thấy cột {satisfaction_col}; notebook sẽ chỉ fill các cột sentiment_score/topic.")

print("Cột ID:", id_col)
print("Số cột sentiment_score/topic sẽ fill:", len(sentiment_cols))
print("Các cột sentiment_score/topic:", sentiment_cols)
print("Có fill satisfaction_score không:", satisfaction_col in cols_to_impute)
print("Tổng số cột đưa vào KNN:", len(cols_to_impute))
print("Các cột đưa vào KNN:", cols_to_impute)

Cột ID: review_id
Số cột sentiment_score/topic sẽ fill: 15
Các cột sentiment_score/topic: ['AR', 'AT', 'CS', 'GS', 'HH', 'LV', 'MH', 'NQ', 'OA', 'PE', 'RC', 'RE', 'RP', 'SR', 'VP']
Có fill satisfaction_score không: True
Tổng số cột đưa vào KNN: 16
Các cột đưa vào KNN: ['AR', 'AT', 'CS', 'GS', 'HH', 'LV', 'MH', 'NQ', 'OA', 'PE', 'RC', 'RE', 'RP', 'SR', 'VP', 'satisfaction_score']


In [4]:
# ===== 3. Đổi 0 thành NaN trong TẤT CẢ cột sẽ KNN fill =====
# Bao gồm: sentiment_score tại từng topic và satisfaction_score.
# Vì trong matrix dạng wide, giá trị 0 thường là ô thiếu sau khi pivot/merge điểm.
df_knn = df.copy()

null_before = int(df_knn[cols_to_impute].isna().sum().sum())
zero_before = int((df_knn[cols_to_impute] == 0).sum().sum())

sentiment_null_before = int(df_knn[sentiment_cols].isna().sum().sum())
sentiment_zero_before = int((df_knn[sentiment_cols] == 0).sum().sum())

if satisfaction_col in cols_to_impute:
    satisfaction_null_before = int(df_knn[satisfaction_col].isna().sum())
    satisfaction_zero_before = int((df_knn[satisfaction_col] == 0).sum())
else:
    satisfaction_null_before = 0
    satisfaction_zero_before = 0

df_knn[cols_to_impute] = df_knn[cols_to_impute].replace(0, np.nan)

missing_for_knn = int(df_knn[cols_to_impute].isna().sum().sum())
sentiment_missing_for_knn = int(df_knn[sentiment_cols].isna().sum().sum())
satisfaction_missing_for_knn = (
    int(df_knn[satisfaction_col].isna().sum())
    if satisfaction_col in cols_to_impute
    else 0
)

print("Chuẩn bị dữ liệu KNN")
print("Null ban đầu trong các cột đưa vào KNN:", null_before)
print("Số giá trị 0 đổi thành NaN trong các cột đưa vào KNN:", zero_before)
print("Tổng NaN dùng để KNN fill:", missing_for_knn)
print("- Null sentiment ban đầu:", sentiment_null_before)
print("- Số giá trị 0 đổi thành NaN ở sentiment:", sentiment_zero_before)
print("- Tổng NaN sentiment dùng để KNN fill:", sentiment_missing_for_knn)
print("- Null satisfaction_score ban đầu:", satisfaction_null_before)
print("- Số giá trị 0 đổi thành NaN ở satisfaction_score:", satisfaction_zero_before)
print("- Tổng NaN satisfaction_score dùng để KNN fill:", satisfaction_missing_for_knn)


Chuẩn bị dữ liệu KNN
Null ban đầu trong các cột đưa vào KNN: 0
Số giá trị 0 đổi thành NaN trong các cột đưa vào KNN: 808334
Tổng NaN dùng để KNN fill: 808334
- Null sentiment ban đầu: 0
- Số giá trị 0 đổi thành NaN ở sentiment: 798351
- Tổng NaN sentiment dùng để KNN fill: 798351
- Null satisfaction_score ban đầu: 0
- Số giá trị 0 đổi thành NaN ở satisfaction_score: 9983
- Tổng NaN satisfaction_score dùng để KNN fill: 9983


In [5]:
# ===== 4. KNN Imputation với k = 15 =====
# Fit_transform trên cả sentiment_score/topic và satisfaction_score.
imputer = KNNImputer(n_neighbors=25)

rows_before = df_knn.shape[0]
cols_before = df_knn.shape[1]

# Nếu có cột toàn NaN sau khi đổi 0 -> NaN, KNN không thể học được cột đó.
# Khi đó cần kiểm tra lại dữ liệu gốc hoặc loại cột đó khỏi mô hình.
all_nan_cols = [col for col in cols_to_impute if df_knn[col].isna().all()]
if all_nan_cols:
    raise ValueError(
        "Có cột toàn NaN sau khi đổi 0 thành NaN, KNN không thể fill các cột này: "
        f"{all_nan_cols}"
    )

df_knn[cols_to_impute] = imputer.fit_transform(df_knn[cols_to_impute])

print("KNN fill tất cả cột điểm")
print("Số dòng trước/sau:", rows_before, "/", df_knn.shape[0])
print("Số cột trước/sau:", cols_before, "/", df_knn.shape[1])


KNN fill tất cả cột điểm
Số dòng trước/sau: 60492 / 60492
Số cột trước/sau: 17 / 17


In [6]:
# ===== 5. Kiểm tra kết quả sau khi KNN fill =====
df_filled = df_knn.copy()

missing_after = int(df_filled[cols_to_impute].isna().sum().sum())
zero_after = int((df_filled[cols_to_impute] == 0).sum().sum())

print("Kiểm tra sau KNN fill")
print("Tổng NaN trong các cột đưa vào KNN trước/sau:", missing_for_knn, "/", missing_after)
print("Tổng giá trị 0 trong các cột đưa vào KNN trước/sau:", zero_before, "/", zero_after)

if satisfaction_col in cols_to_impute:
    satisfaction_null_after = int(df_filled[satisfaction_col].isna().sum())
    satisfaction_zero_after = int((df_filled[satisfaction_col] == 0).sum())
    satisfaction_changed_count = int((df[satisfaction_col] != df_filled[satisfaction_col]).sum())

    print("\nKiểm tra satisfaction_score")
    print("Số giá trị 0 satisfaction_score trước/sau:", satisfaction_zero_before, "/", satisfaction_zero_after)
    print("Số dòng satisfaction_score đã thay đổi:", satisfaction_changed_count)
else:
    print("\nKhông có cột satisfaction_score để kiểm tra.")


Kiểm tra sau KNN fill
Tổng NaN trong các cột đưa vào KNN trước/sau: 808334 / 0
Tổng giá trị 0 trong các cột đưa vào KNN trước/sau: 808334 / 0

Kiểm tra satisfaction_score
Số giá trị 0 satisfaction_score trước/sau: 9983 / 0
Số dòng satisfaction_score đã thay đổi: 9983


In [7]:
# ===== 6. Lưu file ra cùng thư mục với file gốc =====
# Giữ nguyên logic đặt tên output như notebook cũ.
output_path = input_path.parent / f"{input_path.stem}_knn25_filled.csv"

df_filled.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Lưu file kết quả")
print("Đã lưu file tại:", output_path.resolve())
print("Kích thước file output:", df_filled.shape)

Lưu file kết quả
Đã lưu file tại: D:\User\Tài liệu học\CCDNCTKHDL\6. Smart-PLS\Matrix_wide_knn25_filled.csv
Kích thước file output: (60492, 17)


In [8]:
# ===== 7. Báo cáo trước/sau fill =====
missing_sentiment_after = int(df_filled[sentiment_cols].isna().sum().sum())
zero_sentiment_after = int((df_filled[sentiment_cols] == 0).sum().sum())

print("Báo cáo sentiment_score/topic")
print("NaN sentiment trước khi fill:", sentiment_missing_for_knn)
print("NaN sentiment sau khi fill:", missing_sentiment_after)
print("Số giá trị 0 sentiment trước/sau:", sentiment_zero_before, "/", zero_sentiment_after)

if satisfaction_col in cols_to_impute:
    print("\nBáo cáo satisfaction_score")
    print("NaN satisfaction_score trước khi fill:", satisfaction_missing_for_knn)
    print("NaN satisfaction_score sau khi fill:", int(df_filled[satisfaction_col].isna().sum()))
    print("Số giá trị 0 satisfaction_score trước/sau:", satisfaction_zero_before, "/", int((df_filled[satisfaction_col] == 0).sum()))

print("\n5 dòng đầu sau khi fill:")
print(df_filled.head())

Báo cáo sentiment_score/topic
NaN sentiment trước khi fill: 798351
NaN sentiment sau khi fill: 0
Số giá trị 0 sentiment trước/sau: 798351 / 0

Báo cáo satisfaction_score
NaN satisfaction_score trước khi fill: 9983
NaN satisfaction_score sau khi fill: 0
Số giá trị 0 satisfaction_score trước/sau: 9983 / 0

5 dòng đầu sau khi fill:
   review_id        AR        AT        CS        GS        HH        LV  \
0          1  0.576579  0.648830  0.918829  1.132388  0.759324  1.172379   
1          2  0.648413  0.586622  0.781734  1.444835  0.895713  0.897699   
2          3  0.678599  0.398043  0.702949  0.853531  0.882713  1.076669   
3          4  0.930841  0.578084  0.875953  1.194851  0.841494  1.076204   
4          5  0.800879  0.271534  0.721436  1.050610  0.654634  1.076255   

         MH        NQ        OA        PE        RC        RE        RP  \
0  0.500301  2.424682  1.266632  0.815838  0.951816  1.180261  0.861958   
1  0.631607  0.818258  1.119841  0.833405  1.039261  1.267161 